In [0]:
%sql
WITH first_purchase AS (
    SELECT
        customer_id,
        MIN(DATE_TRUNC('month', order_date)) AS first_purchase_month
    FROM workspace.default.orders
    GROUP BY customer_id
)

SELECT *
FROM first_purchase
ORDER BY first_purchase_month;

customer_id,first_purchase_month
C0005,null
C0155,null
C0082,null
C0456,null
C0276,null
C0143,null
C0498,null
C0295,null
C0065,null
C0389,null


In [0]:
%sql
WITH first_purchase AS (
    SELECT
        customer_id,
        MIN(DATE_TRUNC('month', order_date)) AS cohort_month
    FROM workspace.default.orders
    GROUP BY customer_id
)

SELECT
    o.customer_id,
    o.order_id,
    DATE_TRUNC('month', o.order_date) AS order_month,
    f.cohort_month
FROM workspace.default.orders o
JOIN first_purchase f
    ON o.customer_id = f.customer_id
ORDER BY
    f.cohort_month,
    order_month;

customer_id,order_id,order_month,cohort_month
C0044,O00946,null,null
C0005,O00303,null,null
C0295,O00283,null,null
C0456,O00514,null,null
C0456,O00124,null,null
C0143,O00657,null,null
C0498,O00866,null,null
C0276,O00749,null,null
C0065,O00045,null,null
C0389,O00939,null,null


In [0]:
%sql
SELECT
    DATE_TRUNC('month', order_date) AS month,
    COUNT(DISTINCT customer_id) AS active_customers
FROM workspace.default.orders
GROUP BY DATE_TRUNC('month', order_date)
ORDER BY month;

month,active_customers
null,41
2025-08-01T00:00:00.000Z,48
2025-09-01T00:00:00.000Z,62
2025-10-01T00:00:00.000Z,74
2025-11-01T00:00:00.000Z,61
2025-12-01T00:00:00.000Z,77
2026-01-01T00:00:00.000Z,74
2026-02-01T00:00:00.000Z,82
2026-03-01T00:00:00.000Z,71
2026-04-01T00:00:00.000Z,67


In [0]:
%sql
SELECT
    customer_id,
    COUNT(DISTINCT order_id) AS total_orders
FROM workspace.default.orders
GROUP BY customer_id
HAVING COUNT(DISTINCT order_id) > 1
ORDER BY total_orders DESC;

customer_id,total_orders
C0212,7
C0494,7
C0017,6
C0340,6
C0300,6
C0422,6
C0146,6
C0287,6
C0080,6
C0252,5


In [0]:
%sql
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM workspace.default.orders
    GROUP BY customer_id
)

SELECT
    COUNT(*) AS repeat_customers
FROM customer_orders
WHERE total_orders > 1;

repeat_customers
286


In [0]:
%sql
WITH customer_orders AS (
    SELECT
        customer_id,
        COUNT(DISTINCT order_id) AS total_orders
    FROM workspace.default.orders
    GROUP BY customer_id
)

SELECT
    COUNT(*) AS one_time_customers
FROM customer_orders
WHERE total_orders = 1;

one_time_customers
135


In [0]:
%sql
WITH monthly_customers AS (
    SELECT DISTINCT
        customer_id,
        DATE_TRUNC('month', order_date) AS month
    FROM workspace.default.orders
),

next_month_customers AS (
    SELECT DISTINCT
        m1.customer_id,
        m1.month
    FROM monthly_customers m1
    JOIN monthly_customers m2
        ON m1.customer_id = m2.customer_id
        AND m2.month = ADD_MONTHS(m1.month, 1)
)

SELECT
    m.month,
    COUNT(DISTINCT m.customer_id) AS customers_this_month,
    COUNT(DISTINCT n.customer_id) AS customers_returned_next_month,

    ROUND(
        COUNT(DISTINCT n.customer_id) * 100.0
        / COUNT(DISTINCT m.customer_id),
        2
    ) AS retention_rate

FROM monthly_customers m

LEFT JOIN next_month_customers n
    ON m.customer_id = n.customer_id
    AND m.month = n.month

GROUP BY m.month
ORDER BY m.month;

month,customers_this_month,customers_returned_next_month,retention_rate
null,41,0,0.00
2025-08-01T00:00:00.000Z,48,5,10.42
2025-09-01T00:00:00.000Z,62,9,14.52
2025-10-01T00:00:00.000Z,74,10,13.51
2025-11-01T00:00:00.000Z,61,5,8.20
2025-12-01T00:00:00.000Z,77,14,18.18
2026-01-01T00:00:00.000Z,74,12,16.22
2026-02-01T00:00:00.000Z,82,14,17.07
2026-03-01T00:00:00.000Z,71,10,14.08
2026-04-01T00:00:00.000Z,67,7,10.45


In [0]:
%sql
SELECT
    customer_id,
    MAX(DATE_TRUNC('month', order_date)) AS last_purchase_month
FROM workspace.default.orders
GROUP BY customer_id
ORDER BY last_purchase_month;

customer_id,last_purchase_month
C0065,null
C0044,null
C0125,null
C0456,null
C0005,null
C0389,null
C0498,null
C0155,null
C0143,null
C0276,null
